# Product Hierarchy — Step 1: Extract Unique Products

Pulls every unique product name from the transactions table and exports a CSV
for you to fill in the hierarchy mapping.

**Steps:**
1. Run All — generates `weekly-analysis/data/products_to_map.csv`
2. Open the CSV, fill in `category`, `subcategory`, `product`, `subproduct`
3. Save it as `weekly-analysis/data/product_hierarchy.csv`
4. Commit to GitHub — the weekly report will use it automatically

In [ ]:
# ── Extract Unique Products ───────────────────────────────────────

import sys
sys.path.append('/Workspace/Users/davidgao734@gmail.com/boba-cafe/POS')

from pipeline.transforms import _get_spark
from pipeline.config import TRANSACTIONS_TABLE
import pandas as pd

spark = _get_spark()

products = (
    spark.table(TRANSACTIONS_TABLE)
    .groupBy("product", "is_topping")
    .agg(
        {"qty": "sum", "revenue": "sum", "order_number": "count"}
    )
    .toPandas()
    .rename(columns={
        "sum(qty)": "total_qty",
        "sum(revenue)": "total_revenue",
        "count(order_number)": "times_sold",
    })
    .sort_values("total_revenue", ascending=False)
    .reset_index(drop=True)
)

print(f"Found {len(products):,} unique products")
products.head(20)

In [ ]:
# ── Auto-flag likely subproducts ──────────────────────────────────
# Products with these patterns are probably variants of a base product

import re

SUBPRODUCT_PATTERNS = [
    r"без шариков",
    r"no balls",
    r"без топпинга",
    r"\(.*\)",  # anything in parentheses
]

def is_likely_subproduct(name):
    if not isinstance(name, str):
        return False
    name_lower = name.lower()
    return any(re.search(p, name_lower) for p in SUBPRODUCT_PATTERNS)

products["likely_subproduct"] = products["product"].apply(is_likely_subproduct)

print(f"Likely subproduct variants: {products['likely_subproduct'].sum()}")
products[products["likely_subproduct"]][["product", "total_qty", "total_revenue"]]

In [ ]:
# ── Export CSV for Manual Mapping ────────────────────────────────

import os

output = products[["product", "is_topping", "likely_subproduct", "total_qty", "total_revenue", "times_sold"]].copy()

# Add empty columns for you to fill in
output.insert(0, "subproduct",    "")  # leave blank for canonical; fill for variants e.g. "без шариков"
output.insert(0, "product_mapped", "") # canonical product name e.g. "Боба Классик"
output.insert(0, "subcategory",   "")  # e.g. "Milk Tea", "Fruit Tea", "Smoothie"
output.insert(0, "category",      "")  # e.g. "Drinks", "Toppings", "Food"

OUT_DIR = "/Workspace/Users/davidgao734@gmail.com/boba-cafe/weekly-analysis/data"
os.makedirs(OUT_DIR, exist_ok=True)
out_path = f"{OUT_DIR}/products_to_map.csv"

output.to_csv(out_path, index=False)
print(f"Saved {len(output):,} products to {out_path}")
print()
print("Next step: open the CSV, fill in category / subcategory / product_mapped / subproduct")
print("Then save as: weekly-analysis/data/product_hierarchy.csv")